# CIC-IDS-2018 Verbose Preprocessing Pipeline


In [1]:
import pandas as pd
import numpy as np
import os
import glob
import warnings
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import LabelEncoder, StandardScaler, Normalizer

warnings.filterwarnings('ignore')



## 1. Sampling & Integration
Load 10,000 rows per file to create a combined dataset.


In [2]:
DATA_DIR = r"c:\Users\PMLS\Desktop\Cases_FYP\CIC-IDS-2018"
all_files = glob.glob(os.path.join(DATA_DIR, "*.csv"))
df_parts = []
nrows_per_file = 10000

print(f"Found {len(all_files)} CSV files. Loading {nrows_per_file} rows from each...")
for file in all_files:
    df_parts.append(pd.read_csv(file, nrows=nrows_per_file, low_memory=False, encoding='utf-8'))

df = pd.concat(df_parts, ignore_index=True)
print(f"Initial Combined Shape: {df.shape}")



Found 10 CSV files. Loading 10000 rows from each...
Initial Combined Shape: (100000, 84)


## 2. Header Cleaning
Filter out rows where the 'Label' column contains the string 'Label' (fixing repeated headers).


In [3]:
df.columns = df.columns.str.strip()

if 'Label' in df.columns:
    df = df[df['Label'] != 'Label']

print(f"Shape after header cleanup: {df.shape}")



Shape after header cleanup: (99999, 84)


## 3. Attack Inventory
Print a list of all unique values in the 'Label' column.


In [4]:
print("Attack Inventory (Unique Labels):")
if 'Label' in df.columns:
    unique_labels = df['Label'].unique()
    for label in unique_labels:
        print(f" - {label}")
else:
    print("WARNING: 'Label' column not found.")



Attack Inventory (Unique Labels):
 - Benign
 - FTP-BruteForce
 - DoS attacks-GoldenEye
 - DoS attacks-SlowHTTPTest
 - DDoS attacks-LOIC-HTTP
 - DDOS attack-LOIC-UDP
 - DDOS attack-HOIC
 - Brute Force -Web
 - Brute Force -XSS
 - SQL Injection
 - Bot


## 4. Numeric Conversion
Convert all columns (except 'Label') to numeric. Convert inf and -inf to NaN.


In [5]:
# Convert all features to numeric
cols = [c for c in df.columns if c != 'Label']
for c in cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# Handle infinite values
df.replace([np.inf, -np.inf], np.nan, inplace=True)

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
object_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"Count of Numeric Columns vs. Object Columns: {len(numeric_cols)} numeric, {len(object_cols)} object")



Count of Numeric Columns vs. Object Columns: 83 numeric, 1 object


## 5. NaN Handling
Drop columns that are > 80% empty and rows where the 'Label' is missing.


In [6]:
# Drop columns > 80% empty
missing_rates = df.isnull().mean()
cols_to_drop = missing_rates[missing_rates > 0.8].index
df.drop(columns=cols_to_drop, inplace=True)

# Drop rows where 'Label' is missing
if 'Label' in df.columns:
    df.dropna(subset=['Label'], inplace=True)
    
print(f"Shape after NaN cleanup: {df.shape}")



Shape after NaN cleanup: (99999, 79)


## 6. Imputation
Use SimpleImputer(strategy='median') and confirm 0 NaNs safely.


In [7]:
X_raw = df.drop(columns=['Label'])
y_raw = df['Label']

imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X_raw)

remaining_nans = np.isnan(X_imputed).sum()
print(f"Imputation complete. Confirming {remaining_nans} remaining NaNs")



Imputation complete. Confirming 0 remaining NaNs


## 7. Variance Threshold
Apply VarianceThreshold(threshold=0.01) and capture structural shifts.


In [8]:
selector = VarianceThreshold(threshold=0.01)
X_sel = selector.fit_transform(X_imputed)

cols_before = X_raw.shape[1]
cols_after = X_sel.shape[1]
print(f"Columns BEFORE threshold: {cols_before}")
print(f"Columns AFTER threshold: {cols_after}")

if cols_after < cols_before * 0.8: # Consider 20% drop as significant 
    remaining_features = [X_raw.columns[i] for i in selector.get_support(indices=True)]
    print("\nColumns dropped significantly! Remaining columns:")
    for f in remaining_features:
        print(f" - {f}")

if cols_after < 10:
    print("\nWARNING: Fewer than 10 features remain after variance thresholding!")



Columns BEFORE threshold: 78
Columns AFTER threshold: 66


## 8. Final Transformation
Apply LabelEncoder, StandardScaler, and Normalizer.


In [9]:
le = LabelEncoder()
y_encoded = le.fit_transform(y_raw)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sel)

normalizer = Normalizer(norm='l2')
X_normalized = normalizer.fit_transform(X_scaled)

print("Final transformations complete!")



Final transformations complete!


## 9. Export & Validation
Export fully transformed data to a single combined CSV.


In [10]:
remaining_feature_names = [X_raw.columns[i] for i in selector.get_support(indices=True)]
df_final = pd.DataFrame(X_normalized, columns=remaining_feature_names)
df_final['Label'] = y_encoded

output_file = "CIC_IDS_2018_Preprocessed_Combined.csv"
print(f"Writing final exported artifact to {output_file}...")
df_final.to_csv(output_file, index=False)
print(f"Export fully complete! Final shape: {df_final.shape}")



Writing final exported artifact to CIC_IDS_2018_Preprocessed_Combined.csv...
Export fully complete! Final shape: (99999, 67)
